In [119]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [120]:
input_csv = "/kaggle/input/randon-tickers/tickers.csv"
train_csv = "train.csv"
test_csv = "test.csv"

period = "60d" # 8d for 1m,  60d for other ms,  730d for others
interval = "5m" #1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 4h, 1d, 5d, 1wk, 1mo, 3mo
prepost = True
random_state = 55

In [121]:
tickers_df = pd.read_csv(input_csv)

ticker_to_company = dict(
    zip(tickers_df["Ticker"], tickers_df["Company"])
)

tickers = tickers_df["Ticker"].dropna().unique().tolist()

In [122]:
train_tickers, test_tickers = train_test_split(
    tickers,
    test_size=0.2,
    random_state=random_state,
    shuffle=True
)

In [123]:
def fetch_ticker_data(ticker):
    try:
        df = yf.Ticker(ticker).history(
            period=period,
            interval=interval,
            prepost=True
        )

        if df.empty:
            return None

        df = df.reset_index()

        df_final = pd.DataFrame({
            "Company": ticker_to_company.get(ticker, "UNKNOWN"),
            "Ticker": ticker,
            "Datetime": df["Datetime"],
            "Open": df["Open"],
            "Close": df["Close"],
            "Ratio": (df["Close"] / df["Open"]) - 1
        })

        return df_final

    except Exception as e:
        print(f"Hata ({ticker}): {e}")
        return None

In [124]:
train_data = []

for ticker in train_tickers:
    print(f"📥 Train çekiliyor: {ticker}")
    df = fetch_ticker_data(ticker)
    if df is not None:
        train_data.append(df)

train_df = pd.concat(train_data, ignore_index=True)

📥 Train çekiliyor: AMD
📥 Train çekiliyor: PYPL
📥 Train çekiliyor: AVGO
📥 Train çekiliyor: ABNB
📥 Train çekiliyor: COST
📥 Train çekiliyor: AMZN
📥 Train çekiliyor: AAPL
📥 Train çekiliyor: TSLA
📥 Train çekiliyor: MSFT
📥 Train çekiliyor: QCOM
📥 Train çekiliyor: BKNG
📥 Train çekiliyor: SBUX
📥 Train çekiliyor: META
📥 Train çekiliyor: ADBE
📥 Train çekiliyor: NFLX
📥 Train çekiliyor: CSCO


In [125]:
test_data = []

for ticker in test_tickers:
    print(f"📥 Test çekiliyor: {ticker}")
    df = fetch_ticker_data(ticker)
    if df is not None:
        test_data.append(df)

test_df = pd.concat(test_data, ignore_index=True)

📥 Test çekiliyor: GOOGL
📥 Test çekiliyor: PEP
📥 Test çekiliyor: NVDA
📥 Test çekiliyor: INTC


In [126]:
train_df.to_csv(train_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("✅ İşlem tamamlandı")
print(f"Train satır sayısı: {len(train_df)}")
print(f"Test satır sayısı: {len(test_df)}")

✅ İşlem tamamlandı
Train satır sayısı: 156049
Test satır sayısı: 40802
